# STFT Chunk Explorer

Run the **Setup** cell once, then repeatedly run the **Next chunk** cell to step through the file chunk by chunk.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor

%matplotlib inline

In [ ]:
# --- Config ---
WAV_FILE      = '../data/static_10m_000.wav'
CHUNK_SIZE    = 8192
NPERSEG       = 512
NOVERLAP      = 256
OUTER_INDICES = slice(0, 4)   # outer-ring channels (1–4)
CHANNEL_NAMES = ['CH1 (outer, -x-y)', 'CH2 (outer, +x-y)',
                 'CH3 (outer, -x+y)', 'CH4 (outer, +x+y)']

# Build generator once — keep it alive between cells
def make_stft_gen():
    loader = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
    proc   = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
    for chunk in proc.process(loader.stream()):
        yield chunk

stft_gen = make_stft_gen()
print('Generator ready — run the next cell to draw each chunk.')

In [ ]:
# Build generator once — keep it alive between cells
def make_stft_gen():
    loader = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
    proc   = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
    for chunk in proc.process(loader.stream()):
        yield chunk

stft_gen = make_stft_gen()
print('Generator ready — run the next cell to draw each chunk.')


In [ ]:
# ── Next chunk ── run this cell repeatedly ──────────────────────────────────
try:
    chunk = next(stft_gen)
except StopIteration:
    print('No more chunks — file exhausted.')
    chunk = None

if chunk is not None:
    mag = np.abs(chunk.magnitudes[OUTER_INDICES])   # (4, n_freqs, n_frames)
    db  = 20 * np.log10(mag + 1e-6)        # convert to dB
    vmin, vmax = db.min(), db.max()

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
    fig.suptitle(
        f'STFT — timestamp={chunk.timestamp:.3f}s  '
        f'| frames={mag.shape[2]}  '
        f'| freqs={chunk.freqs[0]:.0f}–{chunk.freqs[-1]:.0f} Hz',
        fontsize=13
    )

    # Layout: top row = CH3, CH4 (indices 2,3); bottom row = CH1, CH2 (indices 0,1)
    plot_order = [2, 3, 0, 1]
    times_ms = chunk.times * 1000

    for plot_idx, ch_idx in enumerate(plot_order):
        ax = axes.flat[plot_idx]
        im = ax.pcolormesh(
            times_ms,
            chunk.freqs,
            db[ch_idx],
            shading='auto',
            cmap='inferno',
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(CHANNEL_NAMES[ch_idx], fontsize=10)
        ax.set_ylabel('Frequency (Hz)')
        ax.set_xlabel('Time (ms)')
        fig.colorbar(im, ax=ax, label='dB')

    plt.tight_layout()
    plt.show()
    print(f'timestamp={chunk.timestamp:.3f}s  '
          f'magnitudes shape={mag.shape}  '
          f'(channels, freqs, frames)')
